In [1]:
import pandas as pd


In [2]:
raw = pd.read_csv(r"C:\Users\DC\Downloads\teaching_prices_messy.csv")

In [3]:
# Analysing Errors in the Data Set

raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 4.7 KB


In [4]:
raw.isna().sum()

date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64

In [5]:
raw.duplicated().sum()

np.int64(1)

In [6]:
raw["item"].value_counts(dropna=False)

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64

In [7]:
raw.shape

(97, 6)

In [8]:
raw[raw.duplicated(keep=False)]

,date,item,city,price,unit,data_label
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
96,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [9]:
number = pd.to_numeric(raw["price"],errors="coerce")


In [10]:
print("missing before", raw["price"].isna().sum())
print("missing after", number.isna().sum())

missing before 1
missing after 2


In [14]:
# ------------------------------------------------
#        DATA CLEANING
# ------------------------------------------------



# Check which specific rows failed to convert to numeric (due to commas or outliers)


problematic_rows = raw[number.isna()][["date", "item", "city", "price"]]
print("Problematic price rows:")
display(problematic_rows)


Problematic price rows:


,date,item,city,price
5,2026-01-01,Pulse C,City 2,NaN
17,2026-01-15,Staple A,City 2,"1,250"


In [15]:
# creates a copy of the raw dataset

clean = raw.copy()

In [17]:
# Drops exact duplicate rows


clean = clean.drop_duplicates()


# Fixing Whitespace issues in Item Names

clean["item"] = clean["item"].str.strip().str.title()

In [19]:
# Cleanes price column: remove commas, convert to numeric, handle extreme outliers if needed

clean["price"] = pd.to_numeric(clean["price"].astype(str).str.replace(",",""), errors="coerce")


# Converts date column to proper datetime format

clean["date"] = pd.to_datetime(clean["date"])



In [28]:
# Converts extreme unrealistic outliers (like 9999) to NaN

clean.loc[clean["price"] > 2000, "price"] = float("nan")

In [29]:

print("--- BEFORE CLEANING ---")
print("Shape:", raw.shape)
print("Missing prices:", raw["price"].isna().sum())
print("Exact duplicates:", raw.duplicated().sum())




--- BEFORE CLEANING ---
Shape: (97, 6)
Missing prices: 1
Exact duplicates: 1


In [30]:

print("\n---AFTER CLEANING---")
print("Shape:", clean.shape )
print("Missing prices:", clean["price"].isna().sum())
print("Duplicates remaining:", clean.duplicated().sum())
print("Item value counts:", clean["item"].value_counts())





---AFTER CLEANING---
Shape: (96, 6)
Missing prices: 2
Duplicates remaining: 0
Item value counts: item
Oil B          25
Pulse C        24
Sweetener D    24
Staple A       23
Name: count, dtype: int64


In [31]:
# Descriptive statistics of cleaned prices

display(clean[["price"]].describe())

,price
count,94.000000
mean,214.739362
std,134.041494
min,99.500000
25%,150.000000
50%,200.000000
75%,302.500000
max,1250.000000


In [32]:
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        96 non-null     datetime64[us]
 1   item        96 non-null     str           
 2   city        96 non-null     str           
 3   price       94 non-null     float64       
 4   unit        96 non-null     str           
 5   data_label  96 non-null     str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 4.6 KB
